## Imports

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import random
import time 

## Reading in Data

In [3]:
data = pd.read_csv('../Data/exon_ranges_summary.csv')

## Simulation function

Defaults creating a CSV to False. (so don't get a ton of .csv's)

work in progress! 

In [15]:

def get_results_dictionary(species_data, simulation_rounds, genome_size, num_active_te, range_1_start, range_1_end, range_2_start, range_2_end):
     # Dictionary to store the results
        species_results = {
            'Species': species_data['Species'],
            'Beginning Genome Size': genome_size,
            'Exon Start Range': range_1_start,
            'Exon End Range': range_1_end,
            'Non-Coding Start Range': range_2_start,
            'Non-Coding End Range': range_2_end,
            'Active TEs': num_active_te,
            'TEs mobilized': 0,
            'TEs static': 0,
            'TEs in Exons': 0,
            'TEs in Non-Coding': 0,
            'Exon New Size': range_1_end - range_1_start + 1,
            'Non-Coding New Size': range_2_end - range_2_start + 1,
            'Total Genome Growth': 0,
            'Simulation Rounds': simulation_rounds,
            'Total Time': 0
        }
        return species_results


def get_te_lengths(num_active_te, mean_length=5000, std_dev_length=2000):
    te_lengths = np.random.normal(loc=mean_length, scale=std_dev_length, size=num_active_te)
    te_lengths = np.clip(te_lengths, 100, 10000).astype(int)
    return te_lengths


def simulation(data, simulation_rounds=100, num_active_te=1000, te_mobilize_threshold=0.5, num_species=1, to_csv=False):
    # get TE lengths
    te_lengths = get_te_lengths(num_active_te)

    results_list = []
    
    for i in range(num_species):
        species_data = data.iloc[i]
    

        genome_size = species_data['Genome_size']
        initial_genome_size = genome_size
        average_range = species_data['Average Range']

        exon_range_start, exon_range_end = 0, average_range
        nc_range_start, nc_range_end = average_range + 1, genome_size

        results = get_results_dictionary(species_data, simulation_rounds, genome_size, num_active_te, exon_range_start, exon_range_end, nc_range_start, nc_range_end)

        start_time = time.time()

        for _ in range(simulation_rounds): 
            for _ in range(num_active_te):
                te_length = random.choice(te_lengths) # randomly select a TE length from normal distribution
                if random.random() < te_mobilize_threshold:
                    # prob_exon = exon_range_end / genome_size # probability of TE landing in exon: use this?
                    te_position = random.randint(0, genome_size)
                    # print(f'TE position: {te_position}')

                    if te_position <= exon_range_end: # lands in exon (range 1) or random.random() < prob_exon:
                        exon_range_end += te_length # expand range 1
                        nc_range_start = exon_range_end + 1
                        nc_range_end += te_length # expand range 2 
                        results['TEs in Exons'] += 1
                        genome_size += te_length # increment genome size
                    else:
                        nc_range_end += te_length # expand range 2
                        results['TEs in Non-Coding'] += 1 # increment count for non-coding TEs
                        genome_size += te_length # increment genome size
        
                    results['TEs mobilized'] += 1
                else:
                    results['TEs static'] += 1

        end_time = time.time()
        results['Exon New Size'] = exon_range_end - exon_range_start + 1
        results['Non-Coding New Size'] = nc_range_end - nc_range_start + 1
        results['Total Genome Growth'] = results['Exon New Size'] + results['Non-Coding New Size'] - initial_genome_size
        results['Total Time'] = end_time - start_time

        results_list.append(results)

    results_df = pd.DataFrame(results_list) # , index=[0])

    # save to DataFrame
    if to_csv:
        results_df.to_csv('CSV/simulation_results.csv', index=False)
    return results_df

## Run simulation for Sparrow Hawk + Giant Panda

can do other species or all species, just change what is being passed to function. Change csv to true below if you would like to see the .csv

In [16]:
species_df = simulation(data, num_species=4, to_csv=False)
species_df

,Species,Beginning Genome Size,Exon Start Range,Exon End Range,Non-Coding Start Range,Non-Coding End Range,Active TEs,TEs mobilized,TEs static,TEs in Exons,TEs in Non-Coding,Exon New Size,Non-Coding New Size,Total Genome Growth,Simulation Rounds,Total Time
0,Accipiter_nisus.Accipiter_nisus_ver1.0.112,1190649881,0,533365,533366,1190649881,1000,50140,49860,26,50114,685312,1443024668,253060099,100,0.057821
1,Ailuropoda_melanoleuca.ASM200744v2.112,2444060653,0,21379888,21379889,2444060653,1000,50151,49849,449,49702,23681106,2672811045,252431498,100,0.044318
2,Amazona_collaria.ASM394721v1.112,1258720284,0,10120975,10120976,1258720284,1000,50128,49872,405,49723,12167199,1499620425,253067340,100,0.043773
3,Amphilophus_citrinellus.Midas_v5.112,844902565,0,3510573,3510574,844902565,1000,49971,50029,180,49791,4390790,1093157382,252645607,100,0.040843


## View TE lengths